# Project 3, Part 3, Create a graph database in Neo4j for the BART system for Community Detection

University of California, Berkeley

Master of Information and Data Science (MIDS) program

w205 - Fundamentals of Data Engineering




# Included Modules and Packages

Code cell containing your includes for modules and packages

Some starter code is provided

You may change the starter code as needed

You may add as much code and/or as many code cells as you need

In [ ]:
import neo4j

import csv

import math
import numpy as np
import pandas as pd

import psycopg2

# Supporting code

Code cells containing any supporting code, such as connecting to the database, any functions, etc.  

Remember you can freely use any code from the labs. You do not need to cite code from the labs.

Some starter code is provided

You may change the starter code as needed

You may add as much code and/or as many code cells as you need

In [ ]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [ ]:
session = driver.session(database="neo4j")

In [ ]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [ ]:
def my_neo4j_number_nodes_relationships():
    "print the number of nodes and relationships"
   
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    print("-------------------------")
    print("  Nodes:", number_nodes)
    print("  Relationships:", number_relationships)
    print("-------------------------")


In [ ]:
def my_neo4j_create_node(station_name):
    "create a node with label Station"
    
    query = """
    
    CREATE (:Station {name: $station_name})
    
    """
    
    session.run(query, station_name=station_name)
    

In [ ]:
def my_neo4j_create_relationship_one_way(from_station, to_station, weight):
    "create a relationship one way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [ ]:
def my_neo4j_create_relationship_two_way(from_station, to_station, weight):
    "create relationships two way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

# 3.3.1 Wipe out the Neo4j database

Call the function my_neo4j_wipe_out_database() to wipe out the Neo4j database

In [ ]:
my_neo4j_wipe_out_database()

# 3.3.2 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships()  

The output should look similar to this:
```
-------------------------
  Nodes: 0
  Relationships: 0
-------------------------
```

In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 0
  Relationships: 0
-------------------------


# 3.3.5 Query the list of stations and the lines they serve only the orange,red and yellow lines which connect highly populated off campus studen areas



In [ ]:
connection.rollback()

query = """

select station,line
from lines
where line in ('red','orange','yellow')
order by station,line

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    line = row[1]
    
    my_neo4j_create_node(line+' '+station)
#     my_neo4j_create_relationship_one_way('depart ' + station,line+station,0)
#     my_neo4j_create_relationship_one_way(line+station,'arrive ' + station,0)
    

# 3.3.6 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships() 

The output should look similar to this:
```
-------------------------
  Nodes: 214
  Relationships: 228
-------------------------
```

In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 72
  Relationships: 0
-------------------------


# 3.3.7 Query the list of all possible stations and travel times and build relationship based on travel times

In [ ]:
connection.rollback()

# query = """

# select a.station, a.line as from_line, b.line as to_line, s.transfer_time
# from lines a
#      join lines b
#        on a.station = b.station and a.line <> b.line 
#      join stations s
#        on a.station = s.station
# order by 1, 2, 3

# """

query = """

select line,station_1,station_2,travel_time from travel_times join lines on station=station_1

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    line = row[0]
    from_stn = row[1]
    to_stn = row[2]
    travel_time = int(row[3])
    #transfer_time = int(row[3])
 
    my_neo4j_create_relationship_one_way(line+' '+from_stn,line+' '+to_stn,travel_time)
    my_neo4j_create_relationship_one_way(line+' '+to_stn,line+' '+from_stn,travel_time)

    

# 3.3.8 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships() 

The output should look similar to this:
```
-------------------------
  Nodes: 214
  Relationships: 436
-------------------------
```


In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 72
  Relationships: 140
-------------------------


# 3.3.9 Query the list of all segments between each station and its adjoining stations, create a relationship for each segment both ways. Only for red, orange and yellow lines



In [ ]:
connection.rollback()

query = """

select line, station from lines where line in ('red','orange','yellow');

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()
rows
d = {key[1]:[] for key in rows}
for row in rows:
    d[row[1]].append(row[0])
for stn in d.keys():
    for l in range(0,len(d[stn])-1):
        print(d[stn],stn)
        my_neo4j_create_relationship_one_way(d[stn][l]+' '+stn,d[stn][l+1]+' '+stn,0)
        my_neo4j_create_relationship_one_way(d[stn][l+1]+' '+stn,d[stn][l]+' '+stn,0)

['orange', 'red'] Richmond
['orange', 'red'] El Cerrito del Norte
['orange', 'red'] El Cerrito Plaza
['orange', 'red'] North Berkeley
['orange', 'red'] Downtown Berkeley
['orange', 'red'] Ashby
['orange', 'red', 'yellow'] MacArthur
['orange', 'red', 'yellow'] MacArthur
['orange', 'red', 'yellow'] 19th Street
['orange', 'red', 'yellow'] 19th Street
['orange', 'red', 'yellow'] 12th Street
['orange', 'red', 'yellow'] 12th Street
['red', 'yellow'] West Oakland
['red', 'yellow'] Embarcadero
['red', 'yellow'] Montgomery Street
['red', 'yellow'] Powell Street
['red', 'yellow'] Civic Center
['red', 'yellow'] 16th Street Mission
['red', 'yellow'] 24th Street Mission
['red', 'yellow'] Glen Park
['red', 'yellow'] Balboa Park
['red', 'yellow'] Daly City
['red', 'yellow'] Colma
['red', 'yellow'] South San Francisco
['red', 'yellow'] San Bruno
['red', 'yellow'] SFO


# 3.3.10 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships()

The output should look similar to this:



In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 114
  Relationships: 252
-------------------------


In [ ]:
query = """

CALL gds.graph.project('bart_map2', 'Station', 'LINK', {relationshipProperties: 'weight'})

"""

# CALL gds.betweenness.stream()
# YIELD nodeId, score
# RETURN gds.util.asNode(nodeId).name AS name, score as betweenness
# ORDER BY betweenness DESC

my_neo4j_run_query_pandas(query)

,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'Station': {'label': 'Station', 'properties':...","{'LINK': {'orientation': 'NATURAL', 'indexInve...",bart_map2,72,192,18


Louvian Modularity for Community Detection

In [ ]:
query = """

CALL gds.louvain.stream('bart_map2')
YIELD nodeId, communityId, intermediateCommunityIds
RETURN gds.util.asNode(nodeId).name AS name, communityId as community, intermediateCommunityIds as intermediate_community
ORDER BY community ASC

"""

df = my_neo4j_run_query_pandas(query)
df

,name,community,intermediate_community
0,orange 12th Street,5,None
1,red 12th Street,5,None
2,yellow 12th Street,5,None
3,orange 19th Street,5,None
4,red 19th Street,5,None
...,...,...,...
67,yellow Orinda,68,None
68,yellow Pittsburg,68,None
69,yellow Pittsburg Center,68,None
70,yellow Pleasant Hill,68,None


In [ ]:
communits=list(df['community'].unique())

In [ ]:
for c in communits:
    print(df[['name','community']].where(df['community']==c).dropna())

                 name  community
0  orange 12th Street        5.0
1     red 12th Street        5.0
2  yellow 12th Street        5.0
3  orange 19th Street        5.0
4     red 19th Street        5.0
5  yellow 19th Street        5.0
6    orange MacArthur        5.0
7       red MacArthur        5.0
8    yellow MacArthur        5.0
9    yellow Rockridge        5.0
                        name  community
10           red Embarcadero       17.0
11        yellow Embarcadero       17.0
12     red Montgomery Street       17.0
13  yellow Montgomery Street       17.0
14         red Powell Street       17.0
15      yellow Powell Street       17.0
16          red West Oakland       17.0
17       yellow West Oakland       17.0
                           name  community
18                 orange Ashby       25.0
19                    red Ashby       25.0
20     orange Downtown Berkeley       25.0
21        red Downtown Berkeley       25.0
22  orange El Cerrito del Norte       25.0
23     red El Cerri

In [ ]:
def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select distinct s.station, s.latitude, s.longitude from stations as s 
join (select station from lines where line in ('red','orange','yellow')) as l on l.station=s.station
       
"""

df = my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)
df

,station,latitude,longitude
0,Concord,37.973745,-122.029127
1,Lake Merritt,37.797773,-122.266588
2,Berryessa,37.368361,-121.874655
3,Ashby,37.853068,-122.269957
4,Civic Center,37.779861,-122.413498
5,Balboa Park,37.721667,-122.447500
6,Downtown Berkeley,37.869799,-122.268197
7,16th Street Mission,37.764847,-122.420042
8,Hayward,37.669700,-122.087000
9,Lafayette,37.893186,-122.124614


In [ ]:
df.count()

station      46
latitude     46
longitude    46
dtype: int64

In [ ]:
sather_gate_berkeley = (37.870260430419115, -122.25950168579497)

gmaps.figure(center=sather_gate_berkeley, zoom_level=9)

Figure(layout=FigureLayout(height='420px'))

In [ ]:
stud_pop_areas = [(37.8739663,-122.2834392),(37.857735,-122.2728232),
                  (37.8689508,-122.2968266),(37.9154056,-122.301411),(37.8447574,-122.2512573),
                  (37.7175858,-121.4085484)]
dfs = pd.DataFrame(data=stud_pop_areas,columns=['latitude','longitude'])

In [ ]:
dfs[['latitude','longitude']]

,latitude,longitude
0,37.8763,122.2793
1,37.8561,122.2713
2,37.8744,122.2986
3,42.6526,73.7562
4,37.9161,122.3108
5,37.8415,122.2480
6,37.8012,122.2583


In [ ]:
import gmaps
import gmaps.geojson_geometries

from geographiclib.geodesic import Geodesic

fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=8)

df_markers = df[['latitude','longitude']]
heatmap = gmaps.symbol_layer(stud_pop_areas, fill_color='blue')

marker_layer = gmaps.marker_layer(df_markers)
#marker_layer1 = gmaps.marker_layer(dfs[['latitude','longitude']])


fig.add_layer(marker_layer)
fig.add_layer(heatmap)

fig

Figure(layout=FigureLayout(height='420px'))